# 01 — Unified Splits, Scoring-Path Verification, Bulk Scoring (T/F/C)

**Runs on:** Kaggle GPU T4. **Covers:** TASKS P0.3, P1.1, P1.2, P1.3, P1.4.

**Prerequisites**
- Kaggle secret `HF_TOKEN` set (Add-ons → Secrets) with read access to the private adapter + dataset repos.
- The `src/` package built by Claude Code (per ARCHITECTURE.md) pushed to your GitHub repo.
- GPU accelerator enabled (Settings → Accelerator → GPU T4).

**Outputs (persisted):** split manifests + `scores/{T,F,C}.parquet` — pushed to a private HF dataset repo at the end so they survive the session.

**Does NOT touch UNIFIED-TEST.** Test scoring is notebook 03, run only after Phases 2–5 are complete.

In [ ]:
# --- Environment ---
REPO_URL = "https://github.com/HRS0986/PromptDMZ.git"
!git clone -q {REPO_URL} slm_shield
%cd slm_shield
import sys; sys.path.insert(0, "/kaggle/working/slm_shield")  # make `src` importable
# Kaggle/Colab ship a pre-provisioned torch+CUDA. Do NOT `uv sync` here (it would rebuild the
# GPU stack and risk CUDA mismatch). Install the behaviour-critical libs on top of platform torch,
# pinned to the versions declared in pyproject.toml (the same ones your adapters were trained under).
!pip install -q "transformers==4.53.1" "unsloth==2025.7.2" "peft==0.16.0" \
                "trl==0.19.1" "accelerate==1.14.0" "bitsandbytes==0.50.0"
# If Kaggle's preinstalled versions clash, restart the kernel after install and re-run from here.

In [ ]:
# --- Auth + config ---
from kaggle_secrets import UserSecretsClient
import os
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

SEED = 42
ART = "/kaggle/working/artifacts"           # session-local artefact root
ARTIFACT_REPO = "hirushafernando/slm-shield-artifacts"  # private HF dataset repo for persistence
os.makedirs(f"{ART}/scores", exist_ok=True)

In [ ]:
# --- P0.3: build unified splits (CPU) ---
from src.eval.splits import build_unified_splits
# Pulls the three Hub datasets, unions validation -> UNIFIED-VAL and test -> UNIFIED-TEST,
# dedups (incl. cross VAL/TEST benign collisions -> dropped from VAL, logged),
# partitions UNIFIED-VAL into T/F/C stratified by label x category, writes hash manifests.
manifests = build_unified_splits(seed=SEED, out_dir=f"{ART}/manifests", hf_token=HF_TOKEN)
print(manifests.summary())
manifests.assert_no_overlap()          # AC: 0 hash collisions across T/F/C/UNIFIED-TEST
manifests.assert_min_benign_c(100)     # AC: conformal split has enough benign examples

In [ ]:
# --- P1.1: load backbone (4-bit training checkpoint) + 3 resident adapters ---
from src.model_loader import load_model_with_adapters
model, tokenizer = load_model_with_adapters(hf_token=HF_TOKEN)
import torch
print(f"Peak VRAM after load: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")  # expect < ~3 GB

In [ ]:
# --- P1.2: GATE — batched vs sequential numerical agreement ---
from src.scoring import verify_batched_vs_sequential
report = verify_batched_vs_sequential(model, tokenizer, manifests.sample("F", n=200, seed=SEED))
print(report)
assert report["max_abs_prob_diff"] < 1e-3, "STOP: batched path disagrees with sequential — do not proceed"
USE_BATCHED = report["batched_supported"]  # False -> scorer falls back to sequential (no early exit)

In [ ]:
# --- P1.3: legacy generate+parse agreement (records parse-failure rate for the thesis) ---
from src.scoring import legacy_agreement_check
agree = legacy_agreement_check(model, tokenizer, manifests.sample("F", n=500, seed=SEED),
                               dump_dir=f"{ART}/legacy_disagreements")
print(agree)   # expect argmax agreement >= 0.99; inspect dumped disagreements manually

In [ ]:
# --- P1.4: resumable bulk scoring of T, F, C (NOT test) ---
from src.scoring import bulk_score_split
for split in ["T", "F", "C"]:
    out = bulk_score_split(model, tokenizer, manifests, split,
                           out_path=f"{ART}/scores/{split}.parquet",
                           batched=USE_BATCHED, resume=True)   # resume=True skips already-scored ids
    print(split, out)

In [ ]:
# --- Persist artefacts beyond the session: push to private HF dataset repo ---
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=ART, repo_id=ARTIFACT_REPO, repo_type="dataset")
print("Artefacts persisted to", ARTIFACT_REPO)

## Next steps (CPU — no Kaggle GPU needed)
Download the artefact repo locally (or in any CPU session) and run Phases 2–5: calibration → tokenizer stats → fusion → conformal. Only then run notebook 02 (generalist training) and finally notebook 03 (sealed test run).